# Text Classification With RNNs

In this lab, we'll use RNNs to find the author of a text.

The corpus comes from classic works of English-language literature: 9 authors, 2 books per author, with one file per chapter.

## Downloading the Data From a Git Repository

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s classical-authors

## Downloading the spacy Model for English

In [ ]:
spacy_updated = !pip freeze | grep spacy==3
if not spacy_updated:
  !pip install 'spacy >= 3, < 4'
!python -m spacy download en_core_web_sm

## Importing TensorFlow and the Other Required Libraries

In [ ]:
import re
import typing

import matplotlib.pyplot as plt
import numpy
import pathlib
import seaborn
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import tensorflow as tf
import tqdm.notebook

## Preprocessing

- Each sentence is an example
- Every word (or group of words) that is a named entity is replaced by its type (with the first letter tripled to create new "words")

In [ ]:
nlp = spacy.load(
    "en_core_web_sm",
    exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"))
nlp.enable_pipe("senter")


def replace_ners(tokens: typing.Iterable[spacy.tokens.Token]
                 ) -> typing.Iterator[str]:
  result = []
  for token in tokens:
    if token.ent_iob_ == "O":
      yield token.text
    elif token.ent_iob_ == "B":
      prefix = token.ent_type_[0]
      yield f"{prefix}{prefix}{token.ent_type_}"


def get_data(directory: pathlib.Path,
             ) -> typing.Tuple[typing.List[str], typing.List[str]]:
  texts = []
  authors = []
  paths = list(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.notebook.tqdm(zip(paths, nlp.pipe(contents,
                                                          n_process=-1)),
                                      total=len(paths)):

    author = path.parent.parent.name
    for sentence in doc.sents:
      authors.append(author)
      texts.append(" ".join(replace_ners(sentence)))

  return texts, authors


texts, authors = get_data(pathlib.Path("classical-authors"))

label_encoder = sklearn.preprocessing.LabelEncoder()
y = label_encoder.fit_transform(authors)

print("Number of texts and shape of y:", len(texts), y.shape)

X_train_raw, X_test_raw, y_train, y_test = \
    sklearn.model_selection.train_test_split(texts, y, test_size=0.3)

In [ ]:
print(len(X_train_raw), y_train.shape)

## Preparing the Data as Sequences

### Preparing the Dictionary and the Sequences

With the [`tf.keras.preprocessing.text.Tokenizer`](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer) tool:
- Build a dictionary on the `X_train_raw` corpus
- What's the vocabulary size?
- What's the maximum length, in words, of a sentence?

In [ ]:
# Your code here

#### Solution

In [ ]:
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(X_train_raw)

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(s.split()) for s in X_train_raw)
print(f"Vocabulary size: {vocab_size}")
print(f"Length of the longest word sequence: {max_length}")

### Building the Matrix of Index Sequences

With the [`tf.keras.preprocessing.text.Tokenizer`](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer) and [`tf.keras.utils.pad_sequences`](https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences) tools:
- Turn `X_train_raw` and `X_test_raw` into sequences of word indices in a dictionary (``texts_to_sequences()`` function of the ``Tokenizer`` object instantiated previously)
- **Pad** the sequences so that they have a reasonable length to be processed by a bidirectional GRU:
  - We'll use `maxlen = 150`, because beyond that, GRUs start failing to properly carry the information.
- Store the resulting sequences in `X_train_pad` and `X_test_pad`

In [ ]:
# Your code here

#### Solution

In [ ]:
max_length = 150

X_train_tokens = tokenizer.texts_to_sequences(X_train_raw)
X_test_tokens = tokenizer.texts_to_sequences(X_test_raw)

X_train_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_train_tokens, maxlen=max_length, truncating="pre")
X_test_pad = tf.keras.preprocessing.sequence.pad_sequences(
    X_test_tokens, maxlen=max_length, truncating="pre")

In [ ]:
print(f"Shape of the document corpus:  {X_train_pad.shape}")
print(f"First example:  {X_train_pad[0]}")

## Modeling

Build a model with the following characteristics:
  - An embedding layer of size 300
  - A layer of bidirectional GRUs (Bidirectional is a Keras layer):
    - of size `64`
    - an orthogonal initialization of **the** weight matrices
    - a dropout parameter of `0.2` for the forward parts
  - A neural network layer with a `softmax` activation that takes as input the last output of the [`GRU`](https://keras.io/api/layers/recurrent_layers/gru/) layer
  - A **cross-entropy** based loss function
  - The `adam` optimizer
  - `accuracy` as the evaluation metric

In [ ]:
# Your code here

### Solution

In [ ]:
EMBEDDING_DIM = 300

model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Input(shape=max_length))
model.add(tf.keras.layers.Embedding(vocab_size, EMBEDDING_DIM))
model.add(tf.keras.layers.Bidirectional(
    tf.keras.layers.GRU(64, dropout=0.2)))
model.add(tf.keras.layers.Dense(9, activation="softmax"))
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])
model.summary()

## Training

Train your model on `X_train_pad`:
- with batches of size `1024`
- for `10` epochs
- using 30% of the training set for validation

In [ ]:
# Your code here

### Solution

In [ ]:
model.fit(X_train_pad,
          y_train,
          batch_size=1024,
          epochs=10,
          validation_split=0.3)

## Evaluation

Evaluate your model's performance on the test set.

In [ ]:
# Your code here

### Solution

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_pad, y_test, batch_size=1024)
print(f"Test set accuracy: {test_accuracy:.3f}")

## First Conclusions

*Given the results, what can we say about the quality of this training? Justify.*

Your answer here

### Solution

We observe overfitting: train $\approx$ 75% while validation and test $\approx$ 50%

## Using Pre-Trained Word Embeddings

To get better results, you'll use the pre-trained GloVe embeddings, rather than learning specific embeddings as before.

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

In [ ]:
def glove_path(embedding_dim: int) -> pathlib.Path:
  if embedding_dim in {50, 100, 200, 300}:
    return pathlib.Path("glove.6B.{}d.txt".format(embedding_dim))
  else:
    raise ValueError("embedding_dim must be in {50, 100, 200, 300}")

## Reusing the GloVe Word Embeddings

Keras' Embedding layer can be initialized with a weight matrix where row i is the embedding of word i.

After a quick look at the `glove.6B.300d.txt` files:
- Create a layer `embedding_layer` of type [`tensorflow.keras.layers.Embedding`](https://keras.io/api/layers/core_layers/embedding/):
  - Initialize it with the GloVe embeddings.
  - Initialize the words of our vocabulary that aren't in GloVe with the zero vector
  - Use the appropriate flag to prevent these embeddings from changing during training

In [ ]:
# Your code here

### Solution

In [ ]:

EMBEDDING_DIM = 300
embedding_matrix = numpy.zeros((vocab_size, EMBEDDING_DIM))
found = 0
with glove_path(EMBEDDING_DIM).open() as fh:
  for line in fh:
    values = line.split(" ")
    word = values[0]
    if word in tokenizer.word_index:
      found += 1
      coeffs = numpy.array(values[1:], dtype="float32")
      embedding_matrix[tokenizer.word_index[word]] = coeffs

print(f"Using {found} pre-trained embeddings out of {vocab_size} words "
      "in the vocabulary")

embedding_layer = tf.keras.layers.Embedding(vocab_size,
                                            EMBEDDING_DIM,
                                            weights=[embedding_matrix],
                                            input_length=150,
                                            trainable=False)

## Modeling, Training and Evaluation

- Create a model identical to the one of the previous part, except that the embedding layer is the one we just instantiated from GloVe
- Train this model with the same parameters as in the previous part
- Evaluate it on the test data

NB: to get better results than with SVMs, a model is proposed in the solution.

In [ ]:
# Your code here

### Solution

In [ ]:
model = tf.keras.models.Sequential()
model.add(embedding_layer)
model.add(tf.keras.layers.Bidirectional(
    tf.keras.layers.GRU(64,
                        kernel_initializer="orthogonal",
                        recurrent_initializer="orthogonal",
                        dropout=0.2,
                        recurrent_dropout=0.2)))
model.add(tf.keras.layers.Dense(9, activation ="softmax"))

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])
model.summary()

In [ ]:
model.fit(X_train_pad,
          y_train,
          batch_size=1024,
          epochs=20,
          validation_split=0.3)

In [ ]:
model.evaluate(X_test_pad, y_test)

In [ ]:
EMBEDDING_DIM = 300

model = tf.keras.models.Sequential()
model.add(tf.keras.layers.InputLayer((150,)))
model.add(embedding_layer)
model.add(tf.keras.layers.GRU(64))
model.add(tf.keras.layers.Dense(9, activation="softmax"))
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])
model.summary()

In [ ]:
model.fit(sequences, y_train, epochs=10, validation_split=0.3, batch_size=1024)

## A Solution Better Than SVMs

In [ ]:
bidi_gru_params = dict(kernel_initializer="orthogonal",
                       recurrent_initializer="orthogonal",
                       dropout=0.5)

model = tf.keras.models.Sequential()
model.add(embedding_layer)
model.add(tf.keras.layers.Bidirectional(
    tf.keras.layers.GRU(64, **bidi_gru_params, return_sequences=True)))
model.add(tf.keras.layers.Bidirectional(
    tf.keras.layers.GRU(64, **bidi_gru_params)))
model.add(tf.keras.layers.Dense(9, activation="softmax"))

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

model.summary()

In [ ]:
model.fit(X_train_pad,
          y_train,
          batch_size=1024,
          epochs=30,
          validation_split=0.3)

In [ ]:
model.evaluate(X_test_pad, y_test)

## Displaying the Test Confusion Matrix

*Display the confusion matrix (using a heatmap is recommended).*

In [ ]:
# Your code here

### Solution

In [ ]:
y_pred_softmax = model.predict(X_test_pad)
y_pred = numpy.argmax(y_pred_softmax , axis=1)

In [ ]:
conf_mat = sklearn.metrics.confusion_matrix(y_test, y_pred)
seaborn.heatmap(conf_mat,
                annot=True,
                fmt="d",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cmap="rocket_r")
plt.show()